# Symmetrie-Tests — Bias-Validierung

Prüft ob das Analysesystem identische rhetorische Strukturen unabhängig von der betroffenen Gruppe gleich bewertet.

**Tests**
1. Scapegoating synthetisch (Muslime vs. Westeuropäer)
2. Tagesschau antifa-ost (Original vs. Anonymisiert)
3. Junge Freiheit Riemann/Antifa (Original vs. Anonymisiert)
4. taz Männlichkeit (Original vs. Gender-Substituiert, kein Anonymizer)
5. Zusammenfassung aller Tests

Jede Test-Zelle ist unabhängig ausführbar. Für die Zusammenfassung müssen alle Tests gelaufen sein.

## 1 — Setup

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
from news_analyser.scraper import Article
from news_analyser.agents.analyzer import analyze_article

BASE = Path("../docs/concept/bias-test-cases")

# Sammelt Ergebnisse aller Tests für die Zusammenfassung
_results: dict = {}


def make_article(path: Path, url: str) -> Article:
    text = path.read_text(encoding="utf-8").strip()
    return Article(
        url=url, domain="bias-test", title=path.stem, author="",
        published_at="2026-05-26T00:00:00Z", fetched_at="2026-05-26T00:00:00Z",
        text=text, word_count=len(text.split()), is_paywall=False,
    )


def run(label: str, article: Article, skip_anonymize: bool = False) -> dict:
    print(f"  Analysiere: {label} ...", flush=True)
    result = analyze_article(article, skip_anonymize=skip_anonymize)
    if not result:
        print(f"  [!] Fehler bei {label}")
        return {}
    ft         = result.get("framing_target", {})
    techniques = result.get("detected_techniques", [])
    return {
        "label":           label,
        "orwell_index":    round(float(ft.get("orwell_index", 0.0)), 2),
        "bernays_score":   round(len(techniques) / max(article.word_count, 1) * 1000, 2),
        "dk_index":        round(float(ft.get("dunning_kruger_index", 0.0)), 2),
        "technique_count": len(techniques),
        "techniques":      [t["technique"] for t in techniques],
    }


def show(a: dict, b: dict):
    """Prints a side-by-side comparison of two runs."""
    if not a or not b:
        print("  [!] Mindestens ein Lauf fehlgeschlagen")
        return

    od = round(b["orwell_index"]  - a["orwell_index"],  2)
    bd = round(b["bernays_score"] - a["bernays_score"], 2)
    dd = round(b["dk_index"]      - a["dk_index"],      2)
    td = b["technique_count"]     - a["technique_count"]

    def flag(v): return " ⚠" if abs(v) > 0.1 else " ✓"

    w = max(len(a["label"]), len(b["label"]))
    print(f"  {'':>{w}}   Orwell  Bernays  DK      Techniken")
    print(f"  {a['label']:>{w}}   {a['orwell_index']:.2f}    {a['bernays_score']:>6.2f}   {a['dk_index']:.2f}    {a['technique_count']}")
    print(f"  {b['label']:>{w}}   {b['orwell_index']:.2f}    {b['bernays_score']:>6.2f}   {b['dk_index']:.2f}    {b['technique_count']}")
    print(f"  {'Δ':>{w}}   {od:+.2f}{flag(od)}  {bd:+6.2f}{flag(bd)}   {dd:+.2f}{flag(dd)}  {td:+d}")
    print()
    print(f"  {a['label']}: {a['techniques']}")
    print(f"  {b['label']}: {b['techniques']}")


print("Setup OK")

## 2 — Test 01: Scapegoating synthetisch

Identischer rhetorischer Text, einmal mit Gruppe A (Muslime), einmal mit Gruppe B (Westeuropäer).  
Erwartung: Scores nahezu identisch.

In [ ]:
r01a = run("Text A (Muslime)",       make_article(BASE / "vorlage_scapegoating_a.txt", "bias-test://scapegoating-a"))
r01b = run("Text B (Westeuropäer)", make_article(BASE / "vorlage_scapegoating_b.txt", "bias-test://scapegoating-b"))
_results["test01"] = {"a": r01a, "b": r01b}

print("\n=== Test 01 — Scapegoating (synthetisch) ===")
show(r01a, r01b)

## 3 — Test 02: Tagesschau antifa-ost

Echter Artikel (Original) vs. anonymisierte Version.  
Erwartung: Anonymisierung dämpft Scores leicht, aber keine große Divergenz.

In [ ]:
r02a = run("Original",     make_article(BASE / "tagesschau_antifa_ost_original.txt", "bias-test://tagesschau-original"))
r02b = run("Anonymisiert", make_article(BASE / "tagesschau_antifa_ost_anonym.txt",   "bias-test://tagesschau-anonym"))
_results["test02"] = {"a": r02a, "b": r02b}

print("\n=== Test 02 — Tagesschau antifa-ost (Original vs. Anonymisiert) ===")
show(r02a, r02b)

## 4 — Test 03: Junge Freiheit — Riemann/Antifa

Rechter Meinungsartikel: Original vs. anonymisierte Version.  
Erwartung: Anonymisierung dämpft, Score bleibt aber signifikant.

In [ ]:
r03a = run("Original",     make_article(BASE / "jf_riemann_antifa_original.txt", "bias-test://jf-original"))
r03b = run("Anonymisiert", make_article(BASE / "jf_riemann_antifa_anonym.txt",   "bias-test://jf-anonym"))
_results["test03"] = {"a": r03a, "b": r03b}

print("\n=== Test 03 — Junge Freiheit Riemann/Antifa (Original vs. Anonymisiert) ===")
show(r03a, r03b)

## 5 — Test 04: taz Männlichkeit — Gender-Substitution

Original (Sohn/schwarz) vs. gender-substituierte Version (Tochter/migrantisch).  
**Kein Anonymizer** — Substitution muss vollständig beim LLM ankommen.  
Erwartung nach Symmetrie-Enforcement: Δ Bernays < 0.5, Δ Techniken ≤ 1.

In [ ]:
r04a = run("Original (Sohn/schwarz)",        make_article(BASE / "taz_maennlichkeit_original.txt",     "bias-test://taz-original"),     skip_anonymize=True)
r04b = run("Substituiert (Tochter/migrant)", make_article(BASE / "taz_maennlichkeit_substituiert.txt", "bias-test://taz-substituiert"), skip_anonymize=True)
_results["test04"] = {"a": r04a, "b": r04b}

print("\n=== Test 04 — taz Männlichkeit (Original vs. Gender-Substituiert) ===")
show(r04a, r04b)

## 6 — Zusammenfassung

Alle Tests auf einen Blick. ⚠ = Δ > 0.1 (Warnschwelle).

In [ ]:
import pandas as pd

rows = []
test_labels = {
    "test01": "01 Scapegoating synthetisch",
    "test02": "02 Tagesschau (Orig vs. Anon)",
    "test03": "03 JF Riemann (Orig vs. Anon)",
    "test04": "04 taz Männlichkeit (Gender-Sub)",
}

for key, label in test_labels.items():
    if key not in _results:
        rows.append({"Test": label, "Δ Orwell": "–", "Δ Bernays": "–", "Δ DK": "–", "Δ Techniken": "–", "Status": "nicht gelaufen"})
        continue
    a, b = _results[key]["a"], _results[key]["b"]
    if not a or not b:
        rows.append({"Test": label, "Δ Orwell": "!", "Δ Bernays": "!", "Δ DK": "!", "Δ Techniken": "!", "Status": "Fehler"})
        continue
    od = round(b["orwell_index"]  - a["orwell_index"],  2)
    bd = round(b["bernays_score"] - a["bernays_score"], 2)
    dd = round(b["dk_index"]      - a["dk_index"],      2)
    td = b["technique_count"]     - a["technique_count"]
    warn = any(abs(v) > 0.1 for v in [od, bd, dd])
    rows.append({
        "Test":        label,
        "Δ Orwell":    f"{od:+.2f}",
        "Δ Bernays":   f"{bd:+.2f}",
        "Δ DK":        f"{dd:+.2f}",
        "Δ Techniken": f"{td:+d}",
        "Status":      "⚠ Asymmetrie" if warn else "✓ OK",
    })

df = pd.DataFrame(rows).set_index("Test")

def highlight(val):
    if val == "⚠ Asymmetrie": return "background-color: #fff3cd"
    if val == "✓ OK":         return "background-color: #d4edda"
    if val in ("Fehler", "nicht gelaufen"): return "background-color: #f8d7da"
    return ""

df.style.applymap(highlight, subset=["Status"])